# Test Job Title Mapping on Single Company

Run LLM-based job title to O*NET mapping on one company (Capital One, 183 unique titles) to verify the process works.

In [ ]:
import pandas as pd
import subprocess
import json
import re
import time
import os
from collections import defaultdict

In [ ]:
# Input paths (relative)
titles_path = '../extract_unique_occupation_title/company_unique_titles.csv'
taxonomy_path = '../../raw/onet_job_occupation_taxonomy.csv'

# Output
output_file = 'capital_one_tagged.csv'

print(f'Titles: {titles_path}')
print(f'Taxonomy: {taxonomy_path}')
print(f'Output: {output_file}')

In [ ]:
# Load data
df_titles = pd.read_csv(titles_path)
df_taxonomy = pd.read_csv(taxonomy_path)

# Filter Capital One
capital_one = df_titles[df_titles['company_name'] == 'Capital One'].iloc[0]
titles_list = capital_one['unique_titles'].split(';')

# O*NET categories
onet_titles = df_taxonomy['Title'].tolist()

print(f'Capital One unique titles: {len(titles_list)}')
print(f'O*NET categories: {len(onet_titles)}')
print(f'\nFirst 5 titles: {titles_list[:5]}')

In [ ]:
# Test with 5 titles first
test_titles = titles_list[:5]

prompt = f"""Map each job title to the most relevant O*NET category.

Job titles to map:
{json.dumps(test_titles)}

Available O*NET categories:
{json.dumps(onet_titles)}

Return JSON only:
{{"mappings": [{{"raw": "original title", "onet": "O*NET category"}}]}}
"""

# Save prompt to temp file and call claude
prompt_file = 'temp_prompt.txt'
with open(prompt_file, 'w', encoding='utf-8') as f:
    f.write(prompt)

result = subprocess.run(
    f'type {prompt_file} | claude --print -',
    capture_output=True, text=True, shell=True
)

print(result.stdout)

In [ ]:
# Parse result
json_match = re.search(r'\{.*\}', result.stdout, re.DOTALL)
if json_match:
    mappings = json.loads(json_match.group())
    for m in mappings['mappings']:
        print(f"{m['raw'][:50]:50} -> {m['onet']}")

In [ ]:
# Process all titles in batches
all_mappings = []
batch_size = 30
total = len(titles_list)

for i in range(0, total, batch_size):
    batch = titles_list[i:i+batch_size]
    print(f'Processing {i+1}-{min(i+batch_size, total)} / {total}')
    
    prompt = f"""Map each job title to the most relevant O*NET category.

Job titles to map:
{json.dumps(batch)}

Available O*NET categories:
{json.dumps(onet_titles)}

Return JSON only, no explanation:
{{"mappings": [{{"raw": "original title", "onet": "O*NET category"}}]}}
"""
    
    with open('temp_prompt.txt', 'w', encoding='utf-8') as f:
        f.write(prompt)
    
    result = subprocess.run(
        'type temp_prompt.txt | claude --print -',
        capture_output=True, text=True, shell=True
    )
    
    json_match = re.search(r'\{.*\}', result.stdout, re.DOTALL)
    if json_match:
        mappings = json.loads(json_match.group())
        all_mappings.extend(mappings['mappings'])
    else:
        print(f'  Warning: Failed to parse batch {i}')
    
    time.sleep(1)  # rate limit

print(f'\nTotal mapped: {len(all_mappings)}')

In [ ]:
# Group by O*NET tag
grouped = defaultdict(list)
for m in all_mappings:
    grouped[m['onet']].append(m['raw'])

# Create output DataFrame
output_data = []
for onet_tag, titles in grouped.items():
    output_data.append({
        'onet_tag': onet_tag,
        'company_name': 'Capital One',
        'raw_titles': ';'.join(titles)
    })

df_output = pd.DataFrame(output_data)
df_output.to_csv(output_file, index=False)

print(f'Saved! {len(all_mappings)} titles -> {len(df_output)} O*NET categories')

In [ ]:
# Preview output
print(df_output[['onet_tag', 'company_name']].head(10))

In [ ]:
# Cleanup temp file
if os.path.exists('temp_prompt.txt'):
    os.remove('temp_prompt.txt')
    print('Cleaned up temp_prompt.txt')